# 가상환경 자동 설정
이 노트북을 실행하면 `.venv` 가상환경 생성 → 패키지 설치(TensorFlow + **PyTorch CUDA**) → Jupyter 커널 등록 → 커널 자동 변경까지 완료됩니다.

- PyTorch는 `https://download.pytorch.org/whl/cu124` 에서 CUDA 12.4 빌드로 설치됩니다.
- NVIDIA 드라이버가 CUDA 12.4 이상을 지원해야 하며, GPU 미탑재 환경이면 `PYTORCH_INDEX_URL`을 `https://download.pytorch.org/whl/cpu` 로 바꾸세요.

In [1]:
import subprocess, sys, os, json, pathlib

# === 설정 ===
VENV_NAME = ".venv"
KERNEL_NAME = "ai-practice-venv"
KERNEL_DISPLAY = "Python (AI-practice)"
PACKAGES = [
    "tensorflow",
    "matplotlib",
    "numpy",
    "pandas",
    "scikit-learn",
    "ipykernel",
]
# PyTorch는 CUDA 빌드를 별도 인덱스에서 설치 (CPU 빌드 방지)
PYTORCH_PACKAGES = ["torch", "torchvision"]
PYTORCH_INDEX_URL = "https://download.pytorch.org/whl/cu124"

PROJECT_DIR = pathlib.Path.cwd()
VENV_DIR = PROJECT_DIR / VENV_NAME

if os.name == "nt":
    PYTHON = VENV_DIR / "Scripts" / "python.exe"
    PIP = VENV_DIR / "Scripts" / "pip.exe"
else:
    PYTHON = VENV_DIR / "bin" / "python"
    PIP = VENV_DIR / "bin" / "pip"

def run(cmd, desc=""):
    if desc:
        print(f"\n>>> {desc}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        # 마지막 10줄만 출력
        lines = result.stdout.strip().split("\n")
        for line in lines[-10:]:
            print(f"  {line}")
    if result.returncode != 0 and result.stderr.strip():
        print(f"  [ERROR] {result.stderr.strip()[:300]}")
    return result.returncode == 0

print(f"프로젝트 경로: {PROJECT_DIR}")
print(f"가상환경 경로: {VENV_DIR}")
print(f"일반 패키지: {', '.join(PACKAGES)}")
print(f"PyTorch(CUDA) 패키지: {', '.join(PYTORCH_PACKAGES)} (index: {PYTORCH_INDEX_URL})")

프로젝트 경로: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice
가상환경 경로: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv
일반 패키지: tensorflow, matplotlib, numpy, pandas, scikit-learn, ipykernel
PyTorch(CUDA) 패키지: torch, torchvision (index: https://download.pytorch.org/whl/cu124)


In [2]:
# 1. 가상환경 생성
if VENV_DIR.exists():
    print(f"기존 가상환경 발견: {VENV_DIR}")
else:
    print("가상환경 생성 중...")
    run([sys.executable, "-m", "venv", str(VENV_DIR)], "python -m venv")
    print("가상환경 생성 완료")

# Python 경로 확인
assert PYTHON.exists(), f"Python 실행 파일을 찾을 수 없습니다: {PYTHON}"
print(f"venv Python: {PYTHON}")

기존 가상환경 발견: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv
venv Python: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv\Scripts\python.exe


In [3]:
# 2. pip 업그레이드 + 패키지 설치
run([str(PYTHON), "-m", "pip", "install", "--upgrade", "pip"], "pip 업그레이드")
run([str(PIP), "install"] + PACKAGES, f"패키지 설치: {', '.join(PACKAGES)}")

# PyTorch는 CUDA 휠을 강제로 재설치 (기존 CPU 빌드가 깔려있어도 교체)
run(
    [str(PIP), "install", "--upgrade", "--force-reinstall",
     "--index-url", PYTORCH_INDEX_URL] + PYTORCH_PACKAGES,
    f"PyTorch CUDA 설치: {', '.join(PYTORCH_PACKAGES)}"
)
print("\n패키지 설치 완료")


>>> pip 업그레이드

>>> 패키지 설치: tensorflow, matplotlib, numpy, pandas, scikit-learn, ipykernel

>>> PyTorch CUDA 설치: torch, torchvision

패키지 설치 완료


In [4]:
# 3. Jupyter 커널 등록
run(
    [str(PYTHON), "-m", "ipykernel", "install", "--user",
     f"--name={KERNEL_NAME}", f"--display-name={KERNEL_DISPLAY}"],
    "Jupyter 커널 등록"
)
print(f"\n커널 '{KERNEL_DISPLAY}' ({KERNEL_NAME}) 등록 완료")


>>> Jupyter 커널 등록
  Installed kernelspec ai-practice-venv in C:\Users\PDG\AppData\Roaming\jupyter\kernels\ai-practice-venv

커널 'Python (AI-practice)' (ai-practice-venv) 등록 완료


In [ ]:
# 4. 커널/인터프리터 자동 적용
#    (1) 하위 폴더 포함 모든 .ipynb 커널 메타데이터 변경
#    (2) VS Code 워크스페이스 기본 인터프리터를 .venv로 고정 (.py 파일 대응)

# (1) 재귀적으로 모든 노트북 탐색, .venv 내부/체크포인트는 제외
NOTEBOOKS = [
    p for p in PROJECT_DIR.rglob("*.ipynb")
    if VENV_DIR not in p.parents and ".ipynb_checkpoints" not in p.parts
]

changed = []
for nb_path in NOTEBOOKS:
    if nb_path.name == "envSetting.ipynb":
        continue
    try:
        with open(nb_path, "r", encoding="utf-8") as f:
            nb = json.load(f)
    except Exception as e:
        print(f"  [!] {nb_path.relative_to(PROJECT_DIR)} 읽기 실패: {e}")
        continue

    kernel_info = nb.get("metadata", {}).get("kernelspec", {})
    if kernel_info.get("name") == KERNEL_NAME:
        print(f"  {nb_path.relative_to(PROJECT_DIR)} - 이미 설정됨, 건너뜀")
        continue

    nb.setdefault("metadata", {})["kernelspec"] = {
        "display_name": KERNEL_DISPLAY,
        "language": "python",
        "name": KERNEL_NAME,
    }
    with open(nb_path, "w", encoding="utf-8") as f:
        json.dump(nb, f, ensure_ascii=False, indent=1)
    changed.append(str(nb_path.relative_to(PROJECT_DIR)))
    print(f"  {nb_path.relative_to(PROJECT_DIR)} -> 커널 변경 완료")

print(f"\n총 {len(changed)}개 노트북 커널 변경 완료")
if changed:
    print(f"변경된 파일: {', '.join(changed)}")

# (2) VS Code 워크스페이스 인터프리터 고정 — 이 폴더 안의 모든 .py는 venv로 실행됨
vscode_dir = PROJECT_DIR / ".vscode"
vscode_dir.mkdir(exist_ok=True)
settings_path = vscode_dir / "settings.json"

settings = {}
if settings_path.exists():
    try:
        settings = json.loads(settings_path.read_text(encoding="utf-8"))
    except Exception:
        print(f"  [!] 기존 settings.json 파싱 실패, 새로 만듭니다.")
        settings = {}

settings["python.defaultInterpreterPath"] = str(PYTHON)
# 터미널 열 때 venv 자동 활성화
settings["python.terminal.activateEnvironment"] = True

settings_path.write_text(
    json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"\n.vscode/settings.json 갱신 완료")
print(f"  python.defaultInterpreterPath = {PYTHON}")
print("\n노트북/스크립트를 다시 열면 .venv 환경이 자동 적용됩니다.")

In [6]:
# 5. 설치 확인
print("=== 설치 확인 ===")
result = subprocess.run(
    [str(PIP), "list", "--format=columns"],
    capture_output=True, text=True
)
all_packages = PACKAGES + PYTORCH_PACKAGES
for pkg in all_packages:
    name = pkg.replace("-", "_").lower()
    for line in result.stdout.split("\n"):
        if name in line.lower().replace("-", "_"):
            print(f"  {line.strip()}")
            break
    else:
        print(f"  [!] {pkg} - 설치 안 됨")

# PyTorch CUDA 동작 확인
print("\n=== PyTorch CUDA 확인 ===")
check = subprocess.run(
    [str(PYTHON), "-c",
     "import torch; "
     "print('torch:', torch.__version__); "
     "print('cuda available:', torch.cuda.is_available()); "
     "print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"],
    capture_output=True, text=True
)
print(check.stdout.strip() or check.stderr.strip())

print(f"\n=== 완료 ===")
print(f"가상환경: {VENV_DIR}")
print(f"커널: {KERNEL_DISPLAY}")
print("노트북을 닫고 다시 열면 가상환경 커널로 실행됩니다.")

=== 설치 확인 ===
  [!] tensorflow - 설치 안 됨
  [!] matplotlib - 설치 안 됨
  [!] numpy - 설치 안 됨
  [!] pandas - 설치 안 됨
  [!] scikit-learn - 설치 안 됨
  [!] ipykernel - 설치 안 됨
  [!] torch - 설치 안 됨
  [!] torchvision - 설치 안 됨

=== PyTorch CUDA 확인 ===
torch: 2.6.0+cu124
cuda available: True
device: NVIDIA GeForce RTX 3060 Laptop GPU

=== 완료 ===
가상환경: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv
커널: Python (AI-practice)
노트북을 닫고 다시 열면 가상환경 커널로 실행됩니다.
